# PlacePulse AI: Multi-Model Placement Benchmark Training
## Viva & Publication Ready Lab Notebook (02_placement_models_comparison.ipynb)
**Author:** PlacePulse AI Engineering Team  
**Algorithms Evaluated:** Logistic Regression, Random Forest, XGBoost, Decision Tree, Gradient Boosting  
**Evaluation Metrics:** Accuracy, Precision, Recall, F1-Score, ROC-AUC, 5-Fold Cross-Validation, and Confusion Matrix.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import xgboost as xgb

print("ML Benchmark modules loaded!")


### 1. Load Preprocessed Data & Inspect Benchmark JSON


In [ ]:
benchmarks_path = os.path.join("..", "models", "classifier_benchmarks.json")
with open(benchmarks_path, "r") as f:
    benchmarks = json.load(f)

print(f"Winning Production Model: {benchmarks['production_model']}")
print(f"Best Test F1-Score: {benchmarks['best_f1']:.4f}")
pd.DataFrame(benchmarks["leaderboard"]).T[["cv_f1_mean", "test_accuracy", "test_precision", "test_recall", "test_f1", "roc_auc"]]


### 2. Multi-Model ROC Curves Comparison


In [ ]:
data_path = os.path.join("..", "data", "processed", "college_placement_processed.csv")
df = pd.read_csv(data_path)

pipeline = joblib.load(os.path.join("..", "models", "placement_pipeline.joblib"))
X = pipeline.transform(df)
y = df["PlacedOrNot"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

models = {
    "Logistic Regression": LogisticRegression(C=1.0, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42)
}

plt.figure(figsize=(9, 6))
for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label="Random Chance (AUC = 0.5000)")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curves: Multi-Model Placement Benchmark", fontsize=14, fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


### 3. Confusion Matrix of Production Estimator


In [ ]:
best_model = joblib.load(os.path.join("..", "models", "placement_classifier.joblib"))
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Not Placed (0)", "Placed (1)"],
            yticklabels=["Not Placed (0)", "Placed (1)"])
plt.title("Production Model Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Ground Truth")
plt.tight_layout()
plt.show()


### 4. CTC Salary Package Regressor Benchmarks
Evaluation of Ridge Regression vs Random Forest Regressor vs Gradient Boosting.


In [ ]:
reg_benchmarks_path = os.path.join("..", "models", "regressor_benchmarks.json")
with open(reg_benchmarks_path, "r") as f:
    reg_benchmarks = json.load(f)

print(f"Production Regressor: {reg_benchmarks['production_model']}")
pd.DataFrame(reg_benchmarks["leaderboard"]).T[["cv_mae", "test_mae", "test_rmse", "r2_score"]]
